In [1]:
import sys 

assert sys.version_info >= (3,10)

In [2]:
from packaging.version import Version
import torch

assert Version(torch.__version__) >= Version("2.6.0")

In [3]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

device

'cuda'

In [4]:
import matplotlib.pyplot as plt

plt.rc("font", size=14)
plt.rc("legend", fontsize=14)
plt.rc("axes", labelsize=14, titlesize=14)
plt.rc("xtick", labelsize=10)
plt.rc("ytick", labelsize=10)

In [5]:
import deepxde as dde
import numpy as np

SEED = 42
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Using backend: pytorch
Other supported backends: tensorflow.compat.v1, tensorflow, jax, paddle.
paddle supports more examples now and is recommended.


Setting up the backend


In [6]:
dde.config.set_default_float("float64")
print(f"Backend: {dde.backend.backend_name}")


Set the default float type to float64
Backend: pytorch


Exact solution for validation

In [7]:
def exact_solution(x):
    return (x + 1) ** 2

Domain geometry

In [8]:
geom = dde.geometry.Interval(-1, 1)

Define the Left and Right Boundary Conditions

In [9]:
def pde(x, y):
    dy_xx = dde.grad.hessian(y, x, i=0, j=0)
    return dy_xx - 2

def boundary_left(x, on_boundary):
    return on_boundary and np.isclose(x[0], -1)

bc_left = dde.icbc.DirichletBC(geom, lambda x: 0, boundary_left)

def boundary_right(x, on_boudary):
    return on_boudary and np.isclose(x[0], 1)

bc_right = dde.icbc.NeumannBC(geom, lambda x: 4, boundary_right)

Creating data using numpy

In [10]:
observe_x  = np.linspace(-1, 1, 35).reshape(-1, 1)
observe_y = exact_solution(observe_x)
noise = 0.1 * np.random.randn(35, 1)
observe_y = observe_y + noise
observe = dde.icbc.PointSetBC(observe_x, observe_y, component=0)


Combine all data

In [12]:
data = dde.data.PDE(
    geom, 
    pde,
    [bc_left, bc_right, observe],
    num_domain=3000,
    num_boundary=200,
    num_test=500
)

Build a Neural Network and create a Model

In [13]:
net = dde.nn.FNN([1, 256, 128, 64, 1],"tanh", "Glorot uniform")

model = dde.Model(data, net)

Setting Loss Weights

In [14]:
loss_weights = [1.0, 20.0, 1.0, 100.0]
print(f"λ_PDE (Physics):        {loss_weights[0]}")
print(f"λ_BC_left (u(-1)=0):    {loss_weights[1]}")
print(f"λ_BC_right (du/dx=4):   {loss_weights[2]}")
print(f"λ_data (Measurements):  {loss_weights[3]}")

λ_PDE (Physics):        1.0
λ_BC_left (u(-1)=0):    20.0
λ_BC_right (du/dx=4):   1.0
λ_data (Measurements):  100.0


Train the Model with 2 stage optimization
1. Adam Optimizer
2. L-BFGS

In [15]:
print("\nStage 1: Adam Optimizer")
model.compile(
    "adam",
    lr =0.01,
    loss_weights=loss_weights,
    decay=("inverse time", 1000, 0.1)
)
loss_history, train_state = model.train(iterations=2000, display_every=200)

dde.optimizers.config.set_LBFGS_options(max_iter=200)
print("\nStage 2: L-BFGS optinizer (Fine-tuning)")
model.compile("L-BFGS", loss_weights = loss_weights)
loss_history, train_state = model.train(display_every=100)


Stage 1: Adam Optimizer
Compiling model...
'compile' took 3.001143 s

Training model...



/home/ziaur/ziazh/lib/python3.14/site-packages/torch/autograd/graph.py:882: UserWarning: Attempting to run cuBLAS, but there was no current CUDA context! Attempting to set the primary context... (Triggered internally at /pytorch/aten/src/ATen/cuda/CublasHandlePool.cpp:370.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Step      Train loss                                  Test loss                                   Test metric
0         [4.00e+00, 1.19e-01, 1.54e+01, 3.22e+02]    [4.00e+00, 1.19e-01, 1.54e+01, 3.22e+02]    []  
200       [2.71e-02, 2.21e-03, 1.10e-04, 9.58e-01]    [1.42e-02, 2.21e-03, 1.10e-04, 9.58e-01]    []  
400       [2.65e-03, 8.06e-04, 1.45e-04, 9.53e-01]    [2.25e-03, 8.06e-04, 1.45e-04, 9.53e-01]    []  
600       [3.35e-03, 7.29e-04, 1.66e-04, 9.51e-01]    [3.50e-03, 7.29e-04, 1.66e-04, 9.51e-01]    []  
800       [5.28e-03, 2.92e-01, 1.65e-03, 4.78e+00]    [5.13e-03, 2.92e-01, 1.65e-03, 4.78e+00]    []  
1000      [3.46e-03, 7.17e-04, 1.55e-04, 9.51e-01]    [3.64e-03, 7.17e-04, 1.55e-04, 9.51e-01]    []  
1200      [3.19e-03, 7.11e-04, 1.63e-04, 9.51e-01]    [3.38e-03, 7.11e-04, 1.63e-04, 9.51e-01]    []  
1400      [3.13e-03, 7.10e-04, 1.68e-04, 9.51e-01]    [3.33e-03, 7.10e-04, 1.68e-04, 9.51e-01]    []  
1600      [3.11e-03, 7.10e-04, 1.70e-04, 9.51e-01]    [3.32e-03, 7

TypeError: set_LBFGS_options() got an unexpected keyword argument 'max_iter'. Did you mean 'maxiter'?

Creating our own test data

In [17]:
test_points = 15
x_test = np.linspace(-1, 1, test_points).reshape(-1,1)
y_pred = model.predict(x_test)
y_exact  = exact_solution(x_test)
noise = 0.1 * np.random.randn(15, 1)
y_test = y_exact + noise

absolute_error = np.abs(y_test - y_pred)
relative_error = absolute_error / (np.abs(y_test) + 1e-10)
l2_error = np.linalg.norm(y_test - y_pred) / np.linalg.norm(y_test)

u_at_minus1 = model.predict(np.array([[-1.0]]))[0, 0]
u_at_plus1  = model.predict(np.array([[1.0]]))[0, 0]
x_right = np.array([[1.0]])
du_dx_at_1 = model.predict(
    x_right,
    operator=lambda x, y: dde.grad.jacobian(y, x, i=0, j=0)
)[0, 0]

print("📊 PERFORMANCE METRICS")
print(f"L2 Relative Error:      {l2_error:.6f}")
print(f"Max Absolute Error:     {np.max(absolute_error):.6f}")
print(f"Mean Absolute Error:    {np.mean(absolute_error):.6f}")

print("\n📍 BOUNDARY CONDITION CHECK")
print(f"u(-1) = {u_at_minus1:.6f}  (Target: 0.0)")
print(f"du/dx(1) = {du_dx_at_1:.6f}  (Target: 4.0)")


📊 PERFORMANCE METRICS
L2 Relative Error:      0.055507
Max Absolute Error:     0.179737
Mean Absolute Error:    0.092449

📍 BOUNDARY CONDITION CHECK
u(-1) = -0.005951  (Target: 0.0)
du/dx(1) = 3.987184  (Target: 4.0)


Now Extracting True Individual Losses

In [18]:
final_train_losses =model.losshistory.loss_train[-1]
L_pde, L_bc_l, L_bc_r, L_data = final_train_losses

print("📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)")
print(f"L_PDE      : {L_pde:.6e}")
print(f"L_BC_left  : {L_bc_l:.6e}")
print(f"L_BC_right : {L_bc_r:.6e}")
print(f"L_data     : {L_data:.6e}")

print("\n📉 WEIGHTED CONTRIBUTIONS")
weighted = [w * l for w, l in zip(loss_weights, final_train_losses)]
print(f"λ_PDE  × L_PDE      : {weighted[0]:.6e}")
print(f"λ_BC_l × L_BC_left  : {weighted[1]:.6e}")
print(f"λ_BC_r × L_BC_right : {weighted[2]:.6e}")
print(f"λ_data × L_data     : {weighted[3]:.6e}")
print(f"\nTotal Weighted Loss : {sum(weighted):.6e}")


📉 FINAL TRAINING LOSSES (Unweighted - REAL values used by DeepXDE)
L_PDE      : 3.322429e-03
L_BC_left  : 7.081913e-04
L_BC_right : 1.642436e-04
L_data     : 9.514222e-01

📉 WEIGHTED CONTRIBUTIONS
λ_PDE  × L_PDE      : 3.322429e-03
λ_BC_l × L_BC_left  : 1.416383e-02
λ_BC_r × L_BC_right : 1.642436e-04
λ_data × L_data     : 9.514222e+01

Total Weighted Loss : 9.515987e+01
